<a href="https://colab.research.google.com/github/danielsiso461/Lab2/blob/master/studentProj.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [1]:
from google.colab import drive
from pathlib import Path
import json

# Connect Google Drive
drive.mount('/content/drive')

# Path to JSON file
json_path = Path('/content/drive/MyDrive/students.json')

# Load JSON
with open(json_path, 'r', encoding='utf-8') as f:
    students = json.load(f)

print("JSON loaded successfully!")

Mounted at /content/drive
JSON loaded successfully!


In [2]:

import json
import shutil
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files, output

output.enable_custom_widget_manager()

# Find the JSON file in Google Drive
drive_files = [
    Path("/content/drive/MyDrive/students.json"),
    Path("/content/drive/MyDrive/students(1).json")
]

json_path = next((p for p in drive_files if p.is_file()), None)

if json_path is None:
    print("Please upload students.json")
    uploaded = files.upload()
    filename = next(name for name in uploaded if name.endswith(".json"))
    json_path = Path.cwd() / filename

# Load students
with open(json_path, "r", encoding="utf-8") as f:
    students = json.load(f)

# Backup directory
if str(json_path).startswith("/content/drive/MyDrive/"):
    git_dir = Path("/content/drive/MyDrive/GIT")
else:
    git_dir = Path("/content/GIT")

# Form style
field_style = {"description_width": "140px"}
field_layout = widgets.Layout(width="600px")

header = widgets.HTML("""
<div style="background:#263b59;color:white;padding:18px;
            border-radius:10px;text-align:center">
    <h2 style="margin:0">Student Information</h2>
    <p style="margin:5px">Student Management Form</p>
</div>
""")

student_dropdown = widgets.Dropdown(
    options=[
        (f"{s['firstName']} {s['lastName']}", i)
        for i, s in enumerate(students)
    ],
    description="בחרי סטודנט:",
    style=field_style,
    layout=field_layout
)

first_name = widgets.Text(
    description="שם פרטי:",
    disabled=True,
    style=field_style,
    layout=field_layout
)

last_name = widgets.Text(
    description="שם משפחה:",
    disabled=True,
    style=field_style,
    layout=field_layout
)

email = widgets.Text(
    description="מייל בראודה:",
    disabled=True,
    style=field_style,
    layout=field_layout
)

courses = widgets.Textarea(
    description="קורסים:",
    disabled=True,
    rows=4,
    style=field_style,
    layout=field_layout
)

link = widgets.Text(
    description="קישור מעניין:",
    disabled=True,
    style=field_style,
    layout=field_layout
)

favorite_program = widgets.Text(
    description="תוכנית אהובה:",
    placeholder="Enter favorite program",
    style=field_style,
    layout=field_layout
)

update_button = widgets.Button(
    description="עדכון ושמירה",
    button_style="success",
    icon="save",
    layout=widgets.Layout(width="200px", height="40px")
)

message = widgets.Output()

# Display selected student's details
def show_student(change=None):
    student = students[student_dropdown.value]

    first_name.value = student["firstName"]
    last_name.value = student["lastName"]
    email.value = student["email"]
    courses.value = "\n".join(student["courses"])
    link.value = student["interestingLink"]
    favorite_program.value = student.get("favoriteProgram", "")

    with message:
        clear_output()

student_dropdown.observe(show_student, names="value")

# Update JSON and create backup
def update_student(button):
    index = student_dropdown.value

    students[index]["favoriteProgram"] = favorite_program.value.strip()

    with message:
        clear_output()

        try:
            # Save changes
            with open(json_path, "w", encoding="utf-8") as f:
                json.dump(students, f, ensure_ascii=False, indent=4)

            print("Student updated successfully!")
            print(f"Saved to: {json_path}")

        except Exception as e:
            print(f"Error saving JSON: {e}")
            return

        try:
            # Copy to Git backup folder
            git_dir.mkdir(parents=True, exist_ok=True)
            backup_path = git_dir / "students.json"
            shutil.copy2(json_path, backup_path)

            print(f"Backup saved to: {backup_path}")

        except Exception as e:
            print(f"Backup failed: {e}")

update_button.on_click(update_student)

# Display form
form = widgets.VBox(
    [
        header,
        student_dropdown,
        first_name,
        last_name,
        email,
        courses,
        link,
        favorite_program,
        update_button,
        message
    ],
    layout=widgets.Layout(
        width="650px",
        padding="20px",
        border="1px solid #dddddd"
    )
)

show_student()
display(form)
